# Gold fact -- `dbo.fct_sales`

Sales orders with revenue and cost by product, territory, and salesperson

**Grain:** One row per unique sales_order_number representing a single order placed by a reseller, with product ordered, quantity, unit price, total revenue and cost

> GENERATED FILE -- DO NOT EDIT.
Produced by framework/generators/generate_notebooks.py from the project spec set. Edit the spec and regenerate; hand edits are overwritten and will fail the notebook-lint gate.


In [ ]:
# Parameters -- overridden per environment by the deployment pipeline.
# See 05-deployment.yaml `parameterisation`.
# Reads from lh_silver, writes to wh_gold. Both must be
# attached to this notebook; wh_gold must be the DEFAULT so an
# unqualified write cannot land in the wrong item.
target_item = "wh_gold"
source_item = "lh_silver"
environment = "dev"
dq_failure_action = "warn"

import sys
from datetime import datetime

from pyspark.sql import functions as F

from ttfabric.cleansing import RuleContext, get_rule
from ttfabric.quality import DQRunLog

load_id = f"load_{datetime.utcnow():%Y%m%d_%H%M%S}"

def resolve_table(name: str):
    """Resolve a spec table reference to a DataFrame.

    Deliberately UNQUALIFIED, so the read lands in the default lakehouse.

    Rules reference tables in their OWN layer -- enforce_referential_integrity
    against dim_products, recompute_total_from_lines against fct_order_items --
    and those peers live in the item this notebook writes to, not the one it
    reads its source from. Qualifying with source_item sent them to
    lh_bronze.dim_products, which does not and should not exist.

    The single cross-item read, this table's own bronze source, is qualified
    explicitly at the call site instead.
    """
    bare = name.split(".")[-1]
    return spark.read.table(bare)

ctx = RuleContext(
    spark=spark,
    load_id=load_id,
    environment=environment,
    table="fct_sales",
    resolve_table=resolve_table,
    apply_masking=(environment in ("uat", "prod")),
)

dq = DQRunLog(spark, load_id=load_id, layer="gold", table_name="fct_sales")
print(f"load_id={load_id}  environment={environment}  table=fct_sales")

from ttfabric.warehouse import gold_target

gold = gold_target(
    spark,
    warehouse="wh_gold",
    schema="dbo",
    write_mode="warehouse_connector",
)


In [ ]:
# ---- Read silver -------------------------------------------------
df = spark.read.table(f"{source_item}.stg_sales")


In [ ]:
# Join stg_product (inner)
# 
stg_product = spark.read.table(f"{source_item}.stg_product")

# Rows this join discards, captured before it happens.
# on_unmatched: quarantine -- they are a defect, not an
# acceptable loss, so they stay countable and reconcilable
# instead of leaving only a gap in a total.
_orphans = df.join(stg_product.select("product_key"), on="product_key", how="left_anti")
_orphan_count = _orphans.count()

# Written ALWAYS, including when empty. Skipping the write on a
# clean run leaves the previous run's rows in place, and an
# empty result is exactly when someone trusts what they see --
# so a stale table reads as "these orphans are current".
(_orphans
    .withColumn("_quarantined_at", F.current_timestamp())
    .withColumn("_reason", F.lit("no matching product_key in stg_product"))
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("fct_sales_unmatched_product"))
print(f"quarantined {_orphan_count:,} row(s) to fct_sales_unmatched_product")

df = df.join(stg_product, on="product_key", how="inner")


In [ ]:
# Join stg_reseller (inner)
# 
stg_reseller = spark.read.table(f"{source_item}.stg_reseller")

# Rows this join discards, captured before it happens.
# on_unmatched: quarantine -- they are a defect, not an
# acceptable loss, so they stay countable and reconcilable
# instead of leaving only a gap in a total.
_orphans = df.join(stg_reseller.select("reseller_key"), on="reseller_key", how="left_anti")
_orphan_count = _orphans.count()

# Written ALWAYS, including when empty. Skipping the write on a
# clean run leaves the previous run's rows in place, and an
# empty result is exactly when someone trusts what they see --
# so a stale table reads as "these orphans are current".
(_orphans
    .withColumn("_quarantined_at", F.current_timestamp())
    .withColumn("_reason", F.lit("no matching reseller_key in stg_reseller"))
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("fct_sales_unmatched_reseller"))
print(f"quarantined {_orphan_count:,} row(s) to fct_sales_unmatched_reseller")

df = df.join(stg_reseller, on="reseller_key", how="inner")


In [ ]:
# Join stg_salesperson (inner)
# 
stg_salesperson = spark.read.table(f"{source_item}.stg_salesperson")

# Rows this join discards, captured before it happens.
# on_unmatched: quarantine -- they are a defect, not an
# acceptable loss, so they stay countable and reconcilable
# instead of leaving only a gap in a total.
_orphans = df.join(stg_salesperson.select("employee_key"), on="employee_key", how="left_anti")
_orphan_count = _orphans.count()

# Written ALWAYS, including when empty. Skipping the write on a
# clean run leaves the previous run's rows in place, and an
# empty result is exactly when someone trusts what they see --
# so a stale table reads as "these orphans are current".
(_orphans
    .withColumn("_quarantined_at", F.current_timestamp())
    .withColumn("_reason", F.lit("no matching employee_key in stg_salesperson"))
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("fct_sales_unmatched_salesperson"))
print(f"quarantined {_orphan_count:,} row(s) to fct_sales_unmatched_salesperson")

df = df.join(stg_salesperson, on="employee_key", how="inner")


In [ ]:
# Join stg_region (inner)
# 
stg_region = spark.read.table(f"{source_item}.stg_region")

# Rows this join discards, captured before it happens.
# on_unmatched: quarantine -- they are a defect, not an
# acceptable loss, so they stay countable and reconcilable
# instead of leaving only a gap in a total.
_orphans = df.join(stg_region.select("sales_territory_key"), on="sales_territory_key", how="left_anti")
_orphan_count = _orphans.count()

# Written ALWAYS, including when empty. Skipping the write on a
# clean run leaves the previous run's rows in place, and an
# empty result is exactly when someone trusts what they see --
# so a stale table reads as "these orphans are current".
(_orphans
    .withColumn("_quarantined_at", F.current_timestamp())
    .withColumn("_reason", F.lit("no matching sales_territory_key in stg_region"))
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("fct_sales_unmatched_region"))
print(f"quarantined {_orphan_count:,} row(s) to fct_sales_unmatched_region")

df = df.join(stg_region, on="sales_territory_key", how="inner")


In [ ]:
# Business rule: gross_profit
# 
df = df.withColumn("gross_profit", F.expr("""sales_revenue - sales_cost"""))


In [ ]:
# Business rule: gross_margin_pct
# 
df = df.withColumn("gross_margin_pct", F.expr("""CASE WHEN sales_revenue = 0 THEN 0 ELSE (sales_revenue - sales_cost) / sales_revenue END"""))


In [ ]:
# Resolve product_sk from dim_product
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_product"),
    surrogate_key="product_sk",
    lookup_on="product_key",
    dimension_key="product_key",
    dimension_surrogate_key="product_sk",
)


In [ ]:
# Resolve reseller_sk from dim_reseller
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_reseller"),
    surrogate_key="reseller_sk",
    lookup_on="reseller_key",
    dimension_key="reseller_key",
    dimension_surrogate_key="reseller_sk",
)


In [ ]:
# Resolve salesperson_sk from dim_salesperson
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_salesperson"),
    surrogate_key="salesperson_sk",
    lookup_on="employee_key",
    dimension_key="employee_key",
    dimension_surrogate_key="salesperson_sk",
)


In [ ]:
# Resolve region_sk from dim_region
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_region"),
    surrogate_key="region_sk",
    lookup_on="sales_territory_key",
    dimension_key="sales_territory_key",
    dimension_surrogate_key="region_sk",
)


In [ ]:
# ---- Rename to target names --------------------------------------
df = (df
    .withColumnRenamed("sales", "sales_revenue")
    .withColumnRenamed("cost", "sales_cost")
)


In [ ]:
# ---- Write -------------------------------------------------------
final_columns = ['sales_order_number', 'order_date', 'product_sk', 'reseller_sk', 'salesperson_sk', 'region_sk', 'quantity', 'unit_price', 'sales_revenue', 'sales_cost', 'gross_profit', 'gross_margin_pct']
out = df.select(*[c for c in final_columns if c in df.columns])
# Drop existing audit columns before re-adding (silver layer may have them)
for col in ["_built_at", "_load_id", "_processed_at"]:
    if col in out.columns:
        out = out.drop(col)
# Add gold audit columns
out = (out
    .withColumn("_built_at", F.current_timestamp())
    .withColumn("_load_id", F.lit(load_id)))

gold.write(out, "fct_sales")
print(f"wrote {out.count():,} rows to fct_sales")


In [ ]:
# ---- Tests -------------------------------------------------------
#   GOLD-GRAIN-001: One row per sales order
from ttfabric.quality import (assert_unique, assert_not_null,
                         assert_keys_resolve, assert_reconciles)

assert_unique(out, ['sales_order_number'])
assert_not_null(out, ['product_sk', 'reseller_sk', 'salesperson_sk', 'region_sk'])

# not_null is not enough: a failed lookup yields the unknown-member
# key, not a null, so a fact table with every key unresolved passes
# a not-null check while reporting everything against "Unknown".
#
# Optional keys are excluded: an event that has not happened has
# no date, and the unknown member is the correct destination.
assert_keys_resolve(out, ['product_sk', 'reseller_sk', 'salesperson_sk', 'region_sk'])

# Gold must tie back to silver. Compared against the rows that
# actually reached gold -- the inner join legitimately drops
# lines whose header was quarantined, so comparing against all
# of silver would fail for a correct build.
reachable = (spark.read.table(f"{source_item}.stg_sales")
    .join(spark.read.table(f"{source_item}.stg_product")
          .select("product_key"),
          on="product_key", how="inner"))
expected = reachable.agg(F.sum("sales")).collect()[0][0] or 0
actual = out.agg(F.sum("sales_revenue")).collect()[0][0] or 0
assert_reconciles(float(actual), float(expected), 0.01,
                  "fct_sales.sales_revenue vs stg_sales.sales")

dq.record_input(out.count())
dq.record_output(out.count())
dq.flush()
